# Tutorial 07: Empirical Privacy Auditing

**Level**: Advanced  
**Duration**: 45-60 minutes  
**Prerequisites**: Tutorials 01-03, basic understanding of DP guarantees

## Overview

While privacy accounting gives us *theoretical* upper bounds on privacy loss, **privacy auditing** provides *empirical* lower bounds by actually attacking the trained model.

In this tutorial, you'll learn:

1. **Why audit?** - The gap between theory and practice
2. **Membership inference** - How attacks reveal privacy leakage
3. **Epsilon estimation** - Converting attack success to privacy bounds
4. **Attack metrics** - AUROC, TPR@FPR, and accuracy
5. **Bootstrap confidence intervals** - Quantifying uncertainty
6. **Complete workflow** - End-to-end auditing example

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Opaque auditing functions
from opaque.auditing import (
    epsilon_clopper_pearson,
    epsilon_one_run,
    epsilon_raw_counts,
    attack_auroc,
    tpr_at_fpr,
    max_accuracy,
    audit,
    bootstrap,
    BootstrapParams,
)

np.random.seed(42)

## 1. Why Audit?

Privacy accounting tells us the *worst-case* privacy loss. But in practice:

- The actual privacy loss may be much lower
- Implementation bugs can cause higher-than-expected leakage
- We want to validate that our DP implementation is correct

**The goal**: Find the largest epsilon that an attacker can empirically demonstrate.

```
audited_epsilon <= true_epsilon <= theoretical_epsilon
```

If `audited_epsilon > theoretical_epsilon`, there's likely a bug!

## 2. Membership Inference Attacks

The basic idea:
1. Train a model on dataset D
2. For each example x, compute a "membership score" indicating how likely x was in D
3. Use these scores to distinguish training members from non-members

Common scoring functions:
- **Loss**: Training members typically have lower loss
- **Confidence**: Training members typically have higher prediction confidence
- **LiRA**: Likelihood ratio comparing "trained with x" vs "trained without x"

Let's simulate some attack scores:

In [ ]:
# Simulate membership inference attack scores
# In practice, these come from actually training a model and running an attack

n_canaries = 500  # Number of canary examples

# Scenario 1: Good DP (small separation between in/out)
in_scores_good_dp = np.random.normal(loc=0.55, scale=0.3, size=n_canaries)
out_scores_good_dp = np.random.normal(loc=0.45, scale=0.3, size=n_canaries)

# Scenario 2: Weak DP (larger separation)
in_scores_weak_dp = np.random.normal(loc=0.7, scale=0.25, size=n_canaries)
out_scores_weak_dp = np.random.normal(loc=0.3, scale=0.25, size=n_canaries)

# Scenario 3: No DP (very large separation - privacy breach)
in_scores_no_dp = np.random.normal(loc=0.9, scale=0.1, size=n_canaries)
out_scores_no_dp = np.random.normal(loc=0.1, scale=0.1, size=n_canaries)

In [ ]:
# Visualize the score distributions
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

scenarios = [
    ("Good DP (ε≈3)", in_scores_good_dp, out_scores_good_dp),
    ("Weak DP (ε≈8)", in_scores_weak_dp, out_scores_weak_dp),
    ("No DP (ε→∞)", in_scores_no_dp, out_scores_no_dp),
]

for ax, (title, in_s, out_s) in zip(axes, scenarios):
    ax.hist(in_s, bins=30, alpha=0.6, label="In (training)", density=True)
    ax.hist(out_s, bins=30, alpha=0.6, label="Out (test)", density=True)
    ax.set_xlabel("Membership Score")
    ax.set_ylabel("Density")
    ax.set_title(title)
    ax.legend()

plt.tight_layout()
plt.show()

## 3. Epsilon Estimation

The key insight: if a mechanism is (ε, δ)-DP, then for any threshold t:

$$\text{TPR}(t) \leq e^\epsilon \cdot \text{FPR}(t) + \delta$$

Rearranging:

$$\epsilon \geq \log\left(\frac{\text{TPR}(t) - \delta}{\text{FPR}(t)}\right)$$

Opaque provides three methods to estimate epsilon:

In [ ]:
# Compare epsilon estimation methods on the "Weak DP" scenario
in_scores = in_scores_weak_dp
out_scores = out_scores_weak_dp

# Method 1: Clopper-Pearson (conservative, statistical guarantees)
eps_cp = epsilon_clopper_pearson(in_scores, out_scores, significance=0.05, delta=1e-5)

# Method 2: One-run method (Nasr et al. 2023, less conservative)
eps_one = epsilon_one_run(in_scores, out_scores, significance=0.05, delta=1e-5)

# Method 3: Raw counts (point estimate, no confidence interval)
eps_raw = epsilon_raw_counts(in_scores, out_scores, min_count=50, delta=1e-5)

print("Epsilon Estimates (Weak DP scenario):")
print(f"  Clopper-Pearson: {eps_cp:.2f}")
print(f"  One-run:         {eps_one:.2f}")
print(f"  Raw counts:      {eps_raw:.2f}")

In [ ]:
# Compare across all scenarios
print("Epsilon Estimates by Scenario (Clopper-Pearson):")
print("-" * 40)

for name, in_s, out_s in scenarios:
    eps = epsilon_clopper_pearson(in_s, out_s, significance=0.05, delta=1e-5)
    print(f"{name:20s}: ε ≥ {eps:.2f}")

## 4. Attack Utility Metrics

Beyond epsilon, these metrics help understand attack strength:

In [ ]:
# AUROC: Area Under ROC Curve
# 0.5 = random guessing, 1.0 = perfect attack

print("Attack AUROC by Scenario:")
print("-" * 40)

for name, in_s, out_s in scenarios:
    auroc = attack_auroc(in_s, out_s)
    print(f"{name:20s}: AUROC = {auroc:.3f}")

In [ ]:
# TPR at low FPR: Important for real-world attacks where false positives are costly

fprs = [0.001, 0.01, 0.1]

print("TPR at various FPR thresholds (Weak DP scenario):")
print("-" * 40)

for fpr_val in fprs:
    tpr_val = tpr_at_fpr(in_scores_weak_dp, out_scores_weak_dp, fpr=fpr_val)
    print(f"  FPR = {fpr_val:.1%}: TPR = {tpr_val:.3f}")

In [ ]:
# Max accuracy: Best achievable classification accuracy

print("\nMax Accuracy by Scenario:")
print("-" * 40)

for name, in_s, out_s in scenarios:
    acc = max_accuracy(in_s, out_s)
    print(f"{name:20s}: Accuracy = {acc:.1%}")

## 5. The `audit()` Convenience Function

Compute all metrics in one call:

In [ ]:
# Comprehensive audit in one call
result = audit(
    in_scores_weak_dp, 
    out_scores_weak_dp, 
    significance=0.05, 
    delta=1e-5,
    method="clopper_pearson",
)

print("=== Privacy Audit Results ===")
print(f"Epsilon lower bound: {result.epsilon:.2f}")
print(f"Attack AUROC:        {result.auroc:.3f}")
print(f"TPR at 1% FPR:       {result.tpr_at_low_fpr:.3f}")
print(f"Max accuracy:        {result.max_accuracy:.1%}")

In [ ]:
# Results can be unpacked as a tuple
eps, auroc, tpr_low_fpr, acc = audit(in_scores_weak_dp, out_scores_weak_dp)
print(f"Unpacked: ε={eps:.2f}, AUROC={auroc:.3f}")

## 6. Bootstrap Confidence Intervals

Quantify uncertainty in your estimates using bootstrap resampling:

In [ ]:
# Configure bootstrap
params = BootstrapParams.confidence_interval(
    confidence=0.95,
    num_samples=1000,
    bias_correction=True,  # BCa bootstrap for better accuracy
    acceleration=True,
    seed=42,
)

print(f"Bootstrap config: {params}")

In [ ]:
# Bootstrap AUROC
auroc_ci = bootstrap(attack_auroc, in_scores_weak_dp, out_scores_weak_dp, params)

print(f"AUROC: {attack_auroc(in_scores_weak_dp, out_scores_weak_dp):.3f}")
print(f"95% CI: [{auroc_ci[0]:.3f}, {auroc_ci[1]:.3f}]")

In [ ]:
# Bootstrap epsilon (wrap the function to fix parameters)
def eps_fn(in_s, out_s):
    return epsilon_clopper_pearson(in_s, out_s, significance=0.05, delta=1e-5)

eps_ci = bootstrap(eps_fn, in_scores_weak_dp, out_scores_weak_dp, params)

print(f"\nEpsilon: {eps_fn(in_scores_weak_dp, out_scores_weak_dp):.2f}")
print(f"95% CI: [{eps_ci[0]:.2f}, {eps_ci[1]:.2f}]")

## 7. Complete Auditing Workflow

Here's how to put it all together for a real DP model:

In [ ]:
def full_privacy_audit(in_scores, out_scores, theoretical_epsilon, delta=1e-5):
    """Run a complete privacy audit and compare to theoretical bounds."""
    
    # 1. Run comprehensive audit
    result = audit(in_scores, out_scores, significance=0.05, delta=delta)
    
    # 2. Compute confidence intervals
    params = BootstrapParams.confidence_interval(
        confidence=0.95, num_samples=1000, 
        bias_correction=True, acceleration=True, seed=42
    )
    
    def eps_fn(i, o):
        return epsilon_clopper_pearson(i, o, significance=0.05, delta=delta)
    
    eps_ci = bootstrap(eps_fn, in_scores, out_scores, params)
    auroc_ci = bootstrap(attack_auroc, in_scores, out_scores, params)
    
    # 3. Report results
    print("=" * 50)
    print("PRIVACY AUDIT REPORT")
    print("=" * 50)
    print(f"\nSample sizes: {len(in_scores)} in, {len(out_scores)} out")
    print(f"Delta: {delta}")
    
    print("\n--- Epsilon Bounds ---")
    print(f"Theoretical upper bound: {theoretical_epsilon:.2f}")
    print(f"Audited lower bound:     {result.epsilon:.2f} (95% CI: [{eps_ci[0]:.2f}, {eps_ci[1]:.2f}])")
    print(f"Gap: {theoretical_epsilon - result.epsilon:.2f}")
    
    print("\n--- Attack Metrics ---")
    print(f"AUROC:         {result.auroc:.3f} (95% CI: [{auroc_ci[0]:.3f}, {auroc_ci[1]:.3f}])")
    print(f"TPR at 1% FPR: {result.tpr_at_low_fpr:.3f}")
    print(f"Max accuracy:  {result.max_accuracy:.1%}")
    
    # 4. Sanity check
    print("\n--- Validation ---")
    if result.epsilon > theoretical_epsilon:
        print("⚠️  WARNING: Audited epsilon exceeds theoretical bound!")
        print("    This may indicate a bug in your DP implementation.")
    else:
        print("✓ Audited epsilon is below theoretical bound (as expected)")
    
    return result

In [ ]:
# Run full audit on "Weak DP" scenario
# Assume theoretical epsilon was 8.0
_ = full_privacy_audit(
    in_scores_weak_dp, 
    out_scores_weak_dp, 
    theoretical_epsilon=8.0,
    delta=1e-5
)

In [ ]:
# Run full audit on "Good DP" scenario
# Assume theoretical epsilon was 3.0
_ = full_privacy_audit(
    in_scores_good_dp, 
    out_scores_good_dp, 
    theoretical_epsilon=3.0,
    delta=1e-5
)

## 8. Comparing Epsilon Methods

When should you use each method?

In [ ]:
# Compare methods across different sample sizes
sample_sizes = [50, 100, 200, 500, 1000]

results = {"CP": [], "One-run": [], "Raw": []}

# Use weak DP scenario as base
in_full = np.random.normal(loc=0.7, scale=0.25, size=2000)
out_full = np.random.normal(loc=0.3, scale=0.25, size=2000)

for n in sample_sizes:
    in_s = in_full[:n]
    out_s = out_full[:n]
    
    results["CP"].append(epsilon_clopper_pearson(in_s, out_s, significance=0.05))
    results["One-run"].append(epsilon_one_run(in_s, out_s, significance=0.05))
    results["Raw"].append(epsilon_raw_counts(in_s, out_s, min_count=10))

# Plot
plt.figure(figsize=(10, 6))
for method, eps_values in results.items():
    plt.plot(sample_sizes, eps_values, marker='o', label=method)

plt.xlabel("Sample Size (canaries per group)")
plt.ylabel("Estimated Epsilon")
plt.title("Epsilon Estimation Methods vs Sample Size")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("\nRecommendation:")
print("- Clopper-Pearson: Default choice, formal statistical guarantees")
print("- One-run: Less conservative with small samples, good for tight bounds")
print("- Raw counts: Quick sanity checks only, not for final results")

## Summary

Key takeaways:

1. **Privacy auditing validates DP implementations** by empirically measuring attack success

2. **Epsilon estimation methods**:
   - `epsilon_clopper_pearson()`: Conservative, formal guarantees (default)
   - `epsilon_one_run()`: Tighter with small samples
   - `epsilon_raw_counts()`: Quick checks only

3. **Attack metrics** provide additional insight:
   - AUROC: Overall attack strength
   - TPR@FPR: Worst-case performance
   - Max accuracy: Easy to interpret

4. **Always report confidence intervals** using `bootstrap()`

5. **Audited epsilon should be ≤ theoretical epsilon**. If not, investigate your implementation!

## Next Steps

- **[Privacy Auditing User Guide](../user-guide/auditing.md)**: Detailed conceptual guide
- **[API Reference](../api/auditing.md)**: Full function documentation
- **Nasr et al. (2023)**: [Tight Auditing of Differentially Private Machine Learning](https://arxiv.org/abs/2305.08846)